# Sanity check: trained model predictions vs ground truth

Pick N random test images. For each: left = ground-truth boxes, right = model predictions. Quickest way to tell if a trained model actually learned the task.

In [ ]:
# --- setup: locate repo, find weights ---
import os, sys
from pathlib import Path

for c in (Path.cwd(), Path.cwd() / 'ml-training', Path.cwd().parent, Path('/content/RU-IEEE-Hackathon/ml-training')):
    if (c / 'data.yaml').exists():
        ML_DIR = c
        break
else:
    raise SystemExit('could not find ml-training')
os.chdir(ML_DIR)

WEIGHTS_CANDIDATES = [
    ML_DIR / 'runs/train/trash_v1/weights/best.pt',
    Path('/content/drive/MyDrive/trash-yolo/trash_v1_best.pt'),
]
WEIGHTS = next((p for p in WEIGHTS_CANDIDATES if p.exists()), None)
if WEIGHTS is None:
    raise SystemExit(f'no weights found. Looked in: {[str(p) for p in WEIGHTS_CANDIDATES]}')
print('weights:', WEIGHTS)
print('ml dir: ', ML_DIR)

In [ ]:
# --- load model, pick test images ---
import random
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

CLASS_NAMES = ['bottle', 'cup', 'can', 'wrapper', 'paper']
N = 8
CONF = 0.25

model = YOLO(str(WEIGHTS))
test_imgs = [p for p in (ML_DIR / 'data/images/test').glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
random.seed(0)
samples = random.sample(test_imgs, min(N, len(test_imgs)))
print(f'{len(samples)} test images selected')

In [ ]:
# --- render ground truth vs predictions side by side ---
def draw_boxes(img, boxes, labels, confs=None, color=(0, 200, 0)):
    img = img.copy()
    h, w = img.shape[:2]
    for i, (cls, cx, cy, bw, bh) in enumerate(boxes):
        x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
        x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        tag = labels[int(cls)] if confs is None else f'{labels[int(cls)]} {confs[i]:.2f}'
        cv2.putText(img, tag, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

def parse_gt(lbl_path):
    if not lbl_path.exists():
        return []
    out = []
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) == 5:
            out.append(tuple(float(x) for x in parts))
    return out

fig, axes = plt.subplots(len(samples), 2, figsize=(12, 4.5 * len(samples)))
if len(samples) == 1:
    axes = [axes]
for row, img_path in enumerate(samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # GT
    gt = parse_gt(ML_DIR / 'data/labels/test' / (img_path.stem + '.txt'))
    gt_img = draw_boxes(img, gt, CLASS_NAMES, color=(0, 200, 0))
    axes[row][0].imshow(gt_img)
    axes[row][0].set_title(f'ground truth ({len(gt)} boxes)')
    axes[row][0].axis('off')

    # Predictions
    results = model.predict(str(img_path), conf=CONF, verbose=False)[0]
    pred_boxes = []
    pred_confs = []
    if results.boxes is not None:
        for b in results.boxes:
            cls_id = int(b.cls[0].item())
            conf = float(b.conf[0].item())
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
            bw, bh = (x2 - x1) / w, (y2 - y1) / h
            pred_boxes.append((cls_id, cx, cy, bw, bh))
            pred_confs.append(conf)
    pred_img = draw_boxes(img, pred_boxes, CLASS_NAMES, confs=pred_confs, color=(255, 100, 0))
    axes[row][1].imshow(pred_img)
    axes[row][1].set_title(f'predictions ({len(pred_boxes)} at conf≥{CONF})')
    axes[row][1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- quick numeric summary on the test split ---
from collections import Counter

gt_total = Counter()
pred_total = Counter()
for img_path in test_imgs:
    for cls, *_ in parse_gt(ML_DIR / 'data/labels/test' / (img_path.stem + '.txt')):
        gt_total[int(cls)] += 1
    results = model.predict(str(img_path), conf=CONF, verbose=False)[0]
    if results.boxes is not None:
        for b in results.boxes:
            pred_total[int(b.cls[0].item())] += 1

print(f"{'class':<10} {'GT':>8} {'pred':>8}")
for i, name in enumerate(CLASS_NAMES):
    print(f'{name:<10} {gt_total[i]:>8} {pred_total[i]:>8}')
print(f"\n{'TOTAL':<10} {sum(gt_total.values()):>8} {sum(pred_total.values()):>8}")